In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import pandas as pd
import matplotlib.pyplot as plt
import tqdm

from transformers import AutoTokenizer

In [18]:
metrics = pd.read_csv("training_metrics.csv")

In [ ]:
# plot the metrics
plt.figure(figsize=(12, 8))
plt.subplot(2, 2, 1)
plt.plot(metrics['mse'])
plt.title('MSE Loss')
plt.subplot(2, 2, 2)
plt.plot(metrics['l1'])
plt.title('L1 Norm of Activations')
plt.subplot(2, 2, 3)
plt.plot(metrics['l0'])
plt.title('L0 Norm of Activations')
plt.subplot(2, 2, 4)
plt.plot(metrics['loss'])
plt.title('Total Loss')
plt.tight_layout()
plt.show()

In [7]:
from sae import SparseAutoEncoder
from datasets import load_dataset

# Load tokenizer only (no need for the full model)
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m")

# Load saved SAE
d, m = 512, 512 * 8
sae = SparseAutoEncoder(d=d, m=m)
sae.load_state_dict(torch.load("sae_model.pt", map_location="cpu"))
sae.eval()

# Load activations and process through SAE encoder in batches
# Full encoder output would be ~88GB, so we process in chunks
NUM_TEXTS = 50000
ACTIVATIONS_PATH = f"activations/activations_{NUM_TEXTS}.pt"

activations = torch.load(ACTIVATIONS_PATH, map_location="cpu")
n_tokens = activations.shape[0]
BATCH_SIZE = 4096

# We only need top-k per feature, so we track running top-k instead of storing everything
K = 20  # track top-K activating tokens per feature
top_vals = torch.zeros(m, K)
top_idxs = torch.full((m, K), -1, dtype=torch.long)

with torch.no_grad():
    for start in tqdm.trange(0, n_tokens, BATCH_SIZE):
        batch = activations[start:start + BATCH_SIZE]
        feat_acts = sae.encoder(batch)  # (batch, 4096)

        for j in range(feat_acts.shape[1]):
            combined_vals = torch.cat([top_vals[j], feat_acts[:, j]])
            combined_idxs = torch.cat([top_idxs[j], torch.arange(start, start + feat_acts.shape[0])])
            best = combined_vals.topk(K)
            top_vals[j] = best.values
            top_idxs[j] = combined_idxs[best.indices]

del activations

# Re-tokenize the same texts to get token strings
# TransformerLens prepends a BOS token per text, so we must do the same
dataset = load_dataset("openwebtext", split="train", streaming=True)
BOS = tokenizer.bos_token_id

all_tokens = []
all_contexts = []
for i, example in enumerate(tqdm.tqdm(dataset, total=NUM_TEXTS)):
    if i >= NUM_TEXTS:
        break
    text = example["text"][:512]
    token_ids = [BOS] + tokenizer.encode(text, add_special_tokens=False)
    tokens = [tokenizer.decode(tid) for tid in token_ids]
    for t in tokens:
        all_tokens.append(t)
        all_contexts.append(text)

print(f"Total tokens: {len(all_tokens)}, activations: {n_tokens}")
assert len(all_tokens) == n_tokens, f"Mismatch! tokens={len(all_tokens)} vs activations={n_tokens}"
print(f"Tracked top-{K} activations for {m} features")

100%|██████████| 50000/50000 [00:29<00:00, 1716.74it/s]

Total tokens: 5884867, activations: 5884867
Tracked top-20 activations for 4096 features


In [12]:
def inspect_feature(feature_idx, k=10):
    k = min(k, K)
    vals = top_vals[feature_idx, :k]
    idxs = top_idxs[feature_idx, :k]

    print(f"Feature {feature_idx} — top {k} activating tokens:\n")
    for val, idx in zip(vals, idxs):
        idx = idx.item()
        if idx < 0:
            break
        token = all_tokens[idx]
        context = all_contexts[idx]
        print(f"  {val:.3f}  token={repr(token):>15s}   context: {context[:80]}")

# try a few features
inspect_feature(2)
print()
inspect_feature(100)

Feature 2 — top 10 activating tokens:

  0.787  token=    ' simplest'   context: Buckwheat Pancakes Secrets

Larger images

Print Successful buckwheat pancakes i
  0.783  token=    ' equipped'   context: Spark is one of the big data engines that are trending in recent times. One of t
  0.777  token=       ' fried'   context: Low Carb Coconut Flour Donuts have real fried donut flavour. Grain-free and glut
  0.769  token=           ' K'   context: Mortal Kombat X is the next highly anticipated installment in its legendary, cri
  0.764  token=        ' free'   context: It’s Memorial Day weekend, and that means summer is here. To celebrate, the Priv
  0.755  token=       ' short'   context: Al Gore and Bill Clinton, doing what they can for the cause.

Photo by Tim Clary
  0.737  token=     ' limited'   context: This holiday season, Pizza Hut is giving cheese fanatics the ultimate present. F
  0.734  token=         'able'   context: This item is currently Out of Stock Usually restocked with

In [13]:
# Features where the best activation is 0 are dead
alive_mask = top_vals[:, 0] > 0
dead_count = (~alive_mask).sum().item()

# Sort alive features by their top activation (lowest = sparsest)
alive_idxs = torch.where(alive_mask)[0]
sorted_order = top_vals[alive_idxs, 0].argsort()
sparsest_idxs = alive_idxs[sorted_order]

print(f"Dead features: {dead_count} / {m}")
print()
for idx in sparsest_idxs[:10]:
    idx = idx.item()
    # Count non-zero entries in top-K as a proxy
    n_active = (top_vals[idx] > 0).sum().item()
    print(f"Feature {idx}: top activation = {top_vals[idx, 0]:.4f} ({n_active}/{K} top slots active)")
    inspect_feature(idx, k=5)
    print()

Dead features: 181 / 4096

Feature 1710: top activation = 0.0001 (1/20 top slots active)
Feature 1710 — top 5 activating tokens:

  0.000  token=           'of'   context: Two parties are at loggerheads but good relations could be crucial in a hung par

Feature 1301: top activation = 0.0003 (1/20 top slots active)
Feature 1301 — top 5 activating tokens:

  0.000  token=            'Q'   context: [np_storybar title=”Q&A with Sunwing” link=””]

Sunwing Vacations plan to sue th

Feature 242: top activation = 0.0004 (1/20 top slots active)
Feature 242 — top 5 activating tokens:

  0.000  token=      'quality'   context: <br/><div style="line-height:0px;"><object type="application/x-shockwave-flash" 

Feature 1532: top activation = 0.0004 (1/20 top slots active)
Feature 1532 — top 5 activating tokens:

  0.000  token=     ' unknown'   context: Among mammals, autotomy seems to have evolved several times, but is taxonomicall

Feature 3075: top activation = 0.0005 (1/20 top slots active)
Featu

In [11]:
# Rank features by strongest top activation
best_per_feature = top_vals[:, 0]
ranked = best_per_feature.argsort(descending=True)

for rank, idx in enumerate(ranked[20:40]):
    idx = idx.item()
    print(f"=== Rank {rank+1}: Feature {idx} (top activation: {best_per_feature[idx]:.3f}) ===")
    inspect_feature(idx, k=5)
    print()

=== Rank 1: Feature 2210 (top activation: 7.270) ===
Feature 2210 — top 5 activating tokens:

  7.270  token=            '.'   context: Δραματικά είναι τα στοιχεία για την κατάσταση στην αγορά εργασίας στην χώρας μας
  6.982  token=            '.'   context: Self-driving electric cars? Rad!! A Hyperloop train that can get you from New Yo
  6.971  token=            '.'   context: Fox News Channel pundits Greta Van Susteren and Erick Erickson have gone to war 
  6.952  token=            '.'   context: Solomon Islanders mourn death of Eroni Kumana who helped save life of John F. Ke
  6.952  token=            '.'   context: Saudi Arabia's King Salman bin Abdulaziz Al Saud waits to greet U.S. President D

=== Rank 2: Feature 1267 (top activation: 7.239) ===
Feature 1267 — top 5 activating tokens:

  7.239  token=            '.'   context: Δραματικά είναι τα στοιχεία για την κατάσταση στην αγορά εργασίας στην χώρας μας
  7.119  token=            '.'   context: Syracuse, NY -- Disbarred lawye